# ML Feature Engineering for Time Series

**Track:** Traditional AI Domains — Time Series  
**Prerequisites:** TS/01 (Classical Forecasting), ML/03 (Tree Models)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

### 🔍 Socratic Deep Check
*How do you transform a time series into a tabular dataset that XGBoost — which has no concept of "time" — can use to outperform ARIMA?*

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Classical models (ARIMA) work on raw time sequences. ML models (XGBoost, LightGBM) work on **feature tables**. The key skill is transforming temporal data into engineered features: lag values, rolling statistics, calendar features, and Fourier terms — then letting gradient-boosted trees find complex non-linear patterns that ARIMA can't capture.

**Mental Model:** ARIMA reads a book page-by-page (sequential). XGBoost reads a spreadsheet (tabular). To use XGBoost on time series, you must "flatten" the story into spreadsheet rows: each row = one time point, columns = features derived from past values.

**Common Junior Pitfall:** Using `df['target'].shift(-1)` (future value) as a feature. This is feature leakage — the model uses tomorrow's value to predict tomorrow. All features must look BACKWARD in time only.

---

## 📑 Table of Contents

  * [🔍 Socratic Deep Check](#socratic-deep-check)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [Multi-Step Forecasting Strategies](#multi-step-forecasting-strategies)
* [Knowledge Check](#knowledge-check)
  * [Q1: Why does `shift(-1)` cause leakage?](#q1-why-does-shift-1-cause-leakage)
  * [Q2: Why are Fourier terms useful for tree models?](#q2-why-are-fourier-terms-useful-for-tree-models)
* [Practical Practice](#practical-practice)
  * [Exercise 1](#exercise-1)
  * [Exercise 2](#exercise-2)


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm matplotlib

In [ ]:
# Cell 1 — Create time series feature engineering pipeline
import pandas as pd
import numpy as np

np.random.seed(42)
n = 365 * 3
dates = pd.date_range('2022-01-01', periods=n, freq='D')
trend = np.linspace(100, 180, n)
season = 20 * np.sin(2 * np.pi * np.arange(n) / 365)
weekly = 5 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = np.random.normal(0, 5, n)
values = trend + season + weekly + noise + np.where(np.arange(n) > 600, 30, 0)

df = pd.DataFrame({'date': dates, 'revenue': values})
df = df.set_index('date')

def create_ts_features(df, target_col='revenue', lags=[1,7,14,28], windows=[7,14,30]):
    """Transform time series into ML-ready feature table.
    ALL features look BACKWARD only — no data leakage.
    """
    result = df.copy()
    
    # 1. LAG features (past values as predictors)
    for lag in lags:
        result[f'lag_{lag}'] = result[target_col].shift(lag)
    
    # 2. ROLLING statistics (smoothed trends)
    for w in windows:
        result[f'rolling_mean_{w}'] = result[target_col].shift(1).rolling(w).mean()
        result[f'rolling_std_{w}'] = result[target_col].shift(1).rolling(w).std()
    
    # 3. CALENDAR features (encode cyclical patterns)
    result['day_of_week'] = result.index.dayofweek
    result['month'] = result.index.month
    result['day_of_year'] = result.index.dayofyear
    result['is_weekend'] = (result.index.dayofweek >= 5).astype(int)
    result['quarter'] = result.index.quarter
    
    # 4. FOURIER terms (smooth seasonality for tree models)
    for k in [1, 2, 3]:
        result[f'sin_yearly_{k}'] = np.sin(2 * np.pi * k * result.index.dayofyear / 365)
        result[f'cos_yearly_{k}'] = np.cos(2 * np.pi * k * result.index.dayofyear / 365)
    
    # 5. DIFF features (rate of change)
    result['diff_1'] = result[target_col].diff(1)
    result['diff_7'] = result[target_col].diff(7)
    
    return result.dropna()

df_features = create_ts_features(df)
print(f'Feature table: {df_features.shape[0]} rows × {df_features.shape[1]} columns')
print(f'\nFeatures: {list(df_features.columns)}')

In [ ]:
# Cell 2 — Train XGBoost and LightGBM on time series features
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

# Temporal split
target = 'revenue'
features = [c for c in df_features.columns if c != target]

split_date = '2024-07-01'
train = df_features[:split_date]
test = df_features[split_date:]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# XGBoost
xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, verbosity=0)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print(f'XGBoost MAE: {xgb_mae:.2f}')

# Feature importance
importance = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)
print(f'\nTop 10 Features:')
for feat, imp in importance.head(10).items():
    print(f'  {feat:25s} {imp:.4f}')

# Plot
plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values, label='Actual', color='black')
plt.plot(y_test.index, xgb_pred, label=f'XGBoost (MAE={xgb_mae:.1f})', color='forestgreen', linestyle='--')
plt.title('XGBoost Time Series Forecast'); plt.legend(); plt.show()

---
## Multi-Step Forecasting Strategies

Predicting multiple steps ahead is harder than one step:

| Strategy | How | Pros | Cons |
|----------|-----|------|------|
| **Recursive** | Predict t+1, use it to predict t+2, ... | Simple | Errors compound |
| **Direct** | Train separate model for each horizon | No error compounding | N models needed |
| **Multi-output** | Single model predicts [t+1, t+2, ..., t+H] | Single model | Complex output |

---
## Knowledge Check

### Q1: Why does `shift(-1)` cause leakage?
<details><summary>Answer</summary>

`shift(-1)` creates a column where each row contains the NEXT day's value — the exact thing you're trying to predict. The model learns "tomorrow's value is tomorrow's value" = 100% accuracy in training, 0% usefulness in production. Use `shift(1)` (previous day) instead.
</details>

### Q2: Why are Fourier terms useful for tree models?
<details><summary>Answer</summary>

Tree models can only create axis-aligned splits ("day_of_year > 180?"). They can't learn smooth seasonal curves naturally. Fourier terms (sin/cos) encode smooth cyclical patterns as continuous features that trees can split on effectively.
</details>

---
## Practical Practice

### Exercise 1
Add `holiday` features (US public holidays) using the `holidays` library. Does it improve MAE?

### Exercise 2
Implement recursive multi-step forecasting: predict 7 days ahead by feeding each prediction back as the lag_1 feature.

---
**Next →** `03_deep_learning_time_series.ipynb` — LSTMs, Temporal Convolutional Networks, and Transformers